In [42]:
import polars as pl
import pint
import numpy as np

data0 = pl.read_csv("enthalpy_increments.csv")

regex = r"^([A-Za-z]*)(\d*)$"
data0 = data0.with_columns(
    pl.col("Substance").str.extract(regex, 1).alias("Symbol"),
    pl.col("Substance").str.extract(regex, 2).replace("", "1").cast(pl.Int32).alias("Stoichiometry"),
)
data0 = data0.with_columns(
    (pint.Quantity("kJ").m_as("kcal") * pl.col("H(298.15 K)-H(0)") / pl.col("Stoichiometry")).alias("H(298.15 K)-H(0) per atom (kcal/mol)"),
    (pint.Quantity("kJ").m_as("kcal") * pl.col("Uncertainty") / pl.col("Stoichiometry")).alias("Uncertainty per atom (kcal/mol)")
)
data0.filter(pl.col("Substance").str.ends_with("2"))

data = data0.select(symbol="Symbol", enthalpy_thermal_contribution_298K="H(298.15 K)-H(0) per atom (kcal/mol)", enthalpy_thermal_contribution_298K_uncertainty="Uncertainty per atom (kcal/mol)")
data = data.with_columns(pl.col("enthalpy_thermal_contribution_298K_uncertainty").round_sig_figs(1))
max_decimals = int(-np.log10(data["enthalpy_thermal_contribution_298K_uncertainty"].min()))

data = data.with_columns(pl.col("enthalpy_thermal_contribution_298K").round(max_decimals))
data.write_csv("enthalpy_thermal_contribution_298K_CODATA_peratom_kcal.csv")

data

symbol,enthalpy_thermal_contribution_298K,enthalpy_thermal_contribution_298K_uncertainty
str,f64,f64
"""Ag""",1.3731,0.005
"""Al""",1.0851,0.005
"""Ar""",1.4811,0.0002
"""B""",0.2921,0.002
"""Be""",0.4661,0.005
…,…,…
"""Th""",1.5177,0.01
"""Ti""",1.153,0.004
"""U""",1.521,0.005
